# Google ADK Production Eval Loop with FutureAGI

<a href="https://colab.research.google.com/github/future-agi/cookbooks/blob/main/google_adk_eval_loop/Google_ADK_Eval_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Companion notebook for [How to Evaluate Google ADK Agents with FutureAGI](https://futureagi.com/blog/evaluate-google-adk-agents).

This notebook walks through the runnable parts of the 6-step ADK Production Eval Loop:

1. **Instrument** — `traceai-google-adk` + `fi_instrumentation.register()` (runs end-to-end)
2. **Score** — `fi.evals.evaluate()` on the agent's output (runs end-to-end)
3. **Enrich** — `enable_auto_enrichment()` so every `evaluate()` result attaches to the active OTel span (runs end-to-end)
4. **CI gate** — `AgentEvaluator` against `.test.json` (structure shown; meant for pytest, not a notebook)
5. **Simulate** — `fi.simulate` is voice-only (LiveKit). For text-agent scenarios, use ADK's user-simulator. Pointer below.
6. **Optimize** — `fi.opt.optimizers.BayesianSearchOptimizer`. Covered in the dedicated [agent-opt notebook](https://github.com/future-agi/agent-opt/blob/main/examples/FutureAGI_Agent_Optimizer.ipynb).

Estimated runtime: ~3 minutes (most time is Gemini API calls).

## Prerequisites

You'll need:

- **`GOOGLE_API_KEY`**: get one from [Google AI Studio](https://aistudio.google.com/app/apikey).
- **`FI_API_KEY` and `FI_SECRET_KEY`**: from your [FutureAGI dashboard](https://app.futureagi.com/) (Settings → API keys).
- **`FI_BASE_URL`** (optional): defaults to `https://api.futureagi.com`. Override with `http://localhost:8000` if you're testing against a local self-hosted Docker stack.

Free tiers work for both. The notebook runs Gemini 2.5 Flash and one evaluator call against the FutureAGI Turing engine. Total cost is well under one cent.


## Step 0 — Install

In [ ]:
!pip install -q traceai-google-adk ai-evaluation google-adk
print("Installed.")

In [ ]:
# Set credentials. In Colab use getpass so keys aren't echoed.
import os
from getpass import getpass

for var in ("GOOGLE_API_KEY", "FI_API_KEY", "FI_SECRET_KEY"):
    if not os.environ.get(var):
        os.environ[var] = getpass(f"{var}: ")

# Optional: point the SDK at your local Docker stack instead of api.futureagi.com.
# Leave unset to use the FutureAGI cloud default.
os.environ.setdefault("FI_BASE_URL", "https://api.futureagi.com")
print("FI_BASE_URL:", os.environ["FI_BASE_URL"])
print("Credentials set.")


## Step 1 — Instrument with traceAI

`fi_instrumentation.register()` returns an OpenTelemetry tracer provider configured to send spans to FutureAGI. `GoogleADKInstrumentor().instrument()` patches the `google.adk` runtime so every agent invocation, tool call, and Gemini completion is captured as a structured span — no manual `start_as_current_span` calls anywhere.

In [ ]:
from fi_instrumentation import register
from fi_instrumentation.fi_types import ProjectType
from traceai_google_adk import GoogleADKInstrumentor

tracer_provider = register(
    project_name="adk-eval-loop-cookbook",
    project_type=ProjectType.OBSERVE,
)
GoogleADKInstrumentor().instrument(tracer_provider=tracer_provider)
print("Instrumented. Spans will land in the 'adk-eval-loop-cookbook' project in Observe.")

### Define the canonical weather agent

Mirrors the example in `traceai_google_adk` so any future regression is easy to bisect.

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types

def get_weather(city: str) -> dict:
    """Return a fake current weather report for the given city."""
    if city.lower() == "new york":
        return {
            "status": "success",
            "report": (
                "The weather in New York is sunny with a temperature of 25 degrees "
                "Celsius (77 degrees Fahrenheit)."
            ),
        }
    return {"status": "error", "error_message": f"Weather information for '{city}' is not available."}

agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    description="Agent to answer weather questions using tools.",
    instruction="You must use the available tools to find an answer.",
    tools=[get_weather],
)
print(f"Agent ready: {agent.name}")

### Run the agent and capture the response

Open Observe in another tab while this runs. You should see one agent-invocation span with two children: a `get_weather` tool call and a Gemini LLM completion.

In [ ]:
import asyncio

APP_NAME = "adk_eval_loop"
USER_ID = "cookbook_user"
SESSION_ID = "cookbook_session"
USER_QUERY = "What is the weather in New York?"

async def run_agent_once() -> str:
    runner = InMemoryRunner(agent=agent, app_name=APP_NAME)
    await runner.session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID,
    )
    final = ""
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=types.Content(role="user", parts=[types.Part(text=USER_QUERY)]),
    ):
        if event.is_final_response():
            final = event.content.parts[0].text.strip()
    return final

agent_response = asyncio.run(run_agent_once())
print("Final response:\n", agent_response)

## Step 2 — Score the output with `evaluate()`

The unified `evaluate()` function picks the right engine automatically based on the metric name and `model` argument. `model="turing_flash"` routes to FutureAGI's hosted Turing evaluator, which is the fastest path for span-attached scores; for self-hosted use `model="gemini/gemini-2.0-flash"` to route through any LiteLLM-compatible model.

We score `groundedness`: did the agent's response use the tool's actual output as its grounding context, or did it hallucinate?

In [ ]:
from fi.evals import evaluate

# Context = what the tool returned. Output = what the agent told the user.
tool_context = (
    "The weather in New York is sunny with a temperature of 25 degrees Celsius "
    "(77 degrees Fahrenheit)."
)

result = evaluate(
    "groundedness",
    output=agent_response,
    context=tool_context,
    model="turing_flash",
)

# evaluate() catches errors internally and returns EvalResult(status="failed",
# error=...). If we don't check, a failed eval silently looks like score=None.
if getattr(result, "status", "completed") != "completed" or result.score is None:
    raise RuntimeError(
        f"groundedness eval failed: {getattr(result, 'error', None) or result.reason}"
    )

print(f"score:  {result.score}")
print(f"passed: {result.passed}")
print(f"reason: {result.reason}")


## Step 3: Auto-Enrichment (Span-Attached Scores)

`enable_auto_enrichment()` makes every subsequent `evaluate()` call emit an `eval.<metric>` span as a child of the currently-active OpenTelemetry span, with `score`, `reason`, and `latency_ms` attached as span attributes. No `span_id` plumbing required.

**Important:** The eval is a span of its own; for it to share a trace with the ADK agent's run, both calls must happen inside the same active span context. The example below wraps both in a single parent span so the trace tree shows the agent run, its child Gemini and tool spans, and the eval span all together.

Run `enable_auto_enrichment()` once at process startup.


In [ ]:
from fi.evals.otel import enable_auto_enrichment

enable_auto_enrichment()
print("Auto-enrichment enabled. Future evaluate() calls emit eval.<metric> spans.")


In [ ]:
from opentelemetry import trace

# Wrap the ADK run AND the eval in one parent span so they share a trace.
# Without this wrapping, the agent's run_async spans end before evaluate()
# is called, and the eval lands in a separate trace.
tracer = trace.get_tracer("adk-eval-loop-cookbook")

with tracer.start_as_current_span("weather_agent_groundedness_check"):
    agent_response_2 = asyncio.run(run_agent_once())

    result_2 = evaluate(
        "groundedness",
        output=agent_response_2,
        context=tool_context,
        model="turing_flash",
    )

    if getattr(result_2, "status", "completed") != "completed" or result_2.score is None:
        raise RuntimeError(
            f"groundedness eval failed: {getattr(result_2, 'error', None) or result_2.reason}"
        )

print(f"score:  {result_2.score}")
print(
    "Open Observe → adk-eval-loop-cookbook → click the latest trace. "
    "You'll see the weather_agent_groundedness_check parent span with "
    "the agent run and the eval.groundedness span as siblings underneath."
)


## Step 4 — CI gate with `AgentEvaluator` (structure only)

`AgentEvaluator.evaluate()` is meant to run inside pytest, not a notebook — it reads `.test.json` fixtures and a `test_config.json` defining thresholds, then exits non-zero if any threshold breaks. Drop this in `tests/test_agent_eval.py`:

```python
import pytest
from google.adk.evaluation.agent_evaluator import AgentEvaluator

@pytest.mark.asyncio
async def test_weather_agent_quality():
    await AgentEvaluator.evaluate(
        agent_module="weather_agent",
        eval_dataset_file_path_or_dir="tests/fixtures/weather.test.json",
        config_file_path="tests/fixtures/test_config.json",
    )
```

And `weather.test.json`:

```json
{
  "eval_set_id": "weather_basic",
  "eval_cases": [
    {
      "eval_id": "new_york",
      "conversation": [
        {
          "user_content": {"parts": [{"text": "What is the weather in New York?"}]},
          "final_response": {"parts": [{"text": "sunny, 25 degrees Celsius"}]},
          "intermediate_data": {"tool_uses": [{"name": "get_weather", "args": {"city": "New York"}}]}
        }
      ]
    }
  ]
}
```

And `test_config.json`:

```json
{"criteria": {"tool_trajectory_avg_score": 1.0, "response_match_score": 0.7}}
```

Refer to the [ADK evaluation docs](https://google.github.io/adk-docs/evaluate/) for the full criteria list (tool trajectory, ROUGE-1 response match, hallucinations, multi-turn, etc.).

## Steps 5 and 6 — Simulate + optimize

**Simulate.** `fi.simulate` is built around LiveKit and is the right tool for **voice agents** (real-time audio scenarios with personas calling your deployed agent). For **text-agent** scenarios on ADK, use ADK's own user-simulator framework — see the [ADK user-simulator docs](https://google.github.io/adk-docs/) for `multi_turn_task_success_v1` and the related metrics added with the simulator.

**Optimize.** Bayesian search over prompt variants is covered in detail in the dedicated agent-opt notebook:

→ [FutureAGI Agent Optimizer notebook](https://github.com/future-agi/agent-opt/blob/main/examples/FutureAGI_Agent_Optimizer.ipynb)

It runs `BayesianSearchOptimizer` against a failing-trace dataset and produces a versioned prompt your CI gate can promote. The pattern is identical for ADK agents: feed the optimizer the conversations where `groundedness` or `tool_trajectory_avg_score` fell below threshold, pick an evaluator, let it search.

## Wrap up

What you have now:

- An ADK agent fully instrumented with traceAI — every invocation, tool call, and LLM completion lands in Observe as structured spans.
- A `groundedness` score on every agent response, attached to the right span automatically.
- Drop-in patterns for the CI gate, simulate, and optimize steps when you're ready to go past observability into pre-production testing.

The full write-up — including the production-hardening checklist, pitfalls, and the four ADK workflow patterns (sequential / parallel / loop / dynamic-routing) with eval recipes for each — is in the blog post:

→ [How to Evaluate Google ADK Agents with FutureAGI](https://futureagi.com/blog/evaluate-google-adk-agents)

If you hit anything broken in this notebook, please file an issue on [future-agi/cookbooks](https://github.com/future-agi/cookbooks/issues) — the `traceai-google-adk` package version, ADK metric names, and the unified `evaluate()` API surface all move.